In [4]:
import numpy as np, os, sys
import pandas as pd
from tqdm import tqdm

## Load all the data so we can explore it. 
testing_file = '../python_modelBuilding/SEPSISdat_test.pickle'
training_file = '../python_modelBuilding/SEPSISdat_train.pickle'
validate_file = '../python_modelBuilding/SEPSISdat_validate.pickle'
if os.path.isfile(training_file):
  SEPSISdat_train = pd.DataFrame.from_dict(pd.read_pickle(training_file))
print(len(SEPSISdat_train)) # should be n=2138

if os.path.isfile(testing_file):
  SEPSISdat_test = pd.DataFrame.from_dict(pd.read_pickle(testing_file))
print(len(SEPSISdat_test)) # should be n=540

if os.path.isfile(validate_file):
  SEPSISdat_validate = pd.DataFrame.from_dict(pd.read_pickle(validate_file))
  print(len(SEPSISdat_validate)) # should be n=2000, but you don't have this file

2138
540
2000


In [6]:
import time
import evaluate_performance as ep
#from get_sepsis_score_lr import load_sepsis_model, get_sepsis_score ## This doesnt work but replace with your model
from get_sepsis_score_gbm import load_sepsis_model, get_sepsis_score

# Load model.
print('Loading sepsis model...')
model = load_sepsis_model()

# Iterate over files.
print('Predicting sepsis labels...')

def score_cohort(cohort,label):
    trueLabels = cohort.inhospital_mortality
    starttime = time.time()
    # Ignore SepsisLabel column if present.
    column_names = cohort.columns
    if column_names[-1] == 'inhospital_mortality':
      column_names = column_names[:-1]
      cohort = cohort.drop(columns='inhospital_mortality')
  # Make predictions.
    probSepsis,probSepsisLabel=get_sepsis_score(cohort, model)
    elapsed = round(time.time() - starttime)/len(cohort.index)
    util = ep.evaluate_model(trueLabels,probSepsis,model["thresh"],label,elapsed)
    return util
#
np.set_printoptions(legacy='1.25')
util_train = score_cohort(SEPSISdat_train,"Training")
print(util_train)
util_test = score_cohort(SEPSISdat_test,"Testing")
print(util_test)
if os.path.isfile(validate_file):
    util_validate = score_cohort(SEPSISdat_validate,"Evaluation")
    print(util_validate)

Loading sepsis model...
Predicting sepsis labels...
{'AUC': 0.997, 'AUPRC': 0.942, 'Net Benefit': 0.038, 'ECE': 0.0449, 'tp': 89.0, 'fp': 77.0, 'fn': 0.0, 'tn': 1972.0, 'F1': 0.698, 'Sensitivity': 1.0, 'Specificity': 0.962, 'Inference Speed': 0.0, 'Weighted Score': 0.77}
{'AUC': 0.801, 'AUPRC': 0.313, 'Net Benefit': 0.011, 'ECE': 0.0195, 'tp': 9.0, 'fp': 30.0, 'fn': 21.0, 'tn': 480.0, 'F1': 0.261, 'Sensitivity': 0.3, 'Specificity': 0.941, 'Inference Speed': 0.0, 'Weighted Score': 0.27}
{'AUC': 0.75, 'AUPRC': 0.129, 'Net Benefit': 0.004, 'ECE': 0.0046, 'tp': 23.0, 'fp': 143.0, 'fn': 47.0, 'tn': 1787.0, 'F1': 0.195, 'Sensitivity': 0.329, 'Specificity': 0.926, 'Inference Speed': 0.001, 'Weighted Score': 0.17}
